# HDD 23 - Frozen-detector controlled SMART sensitivity

Notebook 22 showed that residual clusters usually collapse into one dominant group and lose most persistence beyond a shuffled control. **Do not put regime IDs into the live anomaly score yet.** This notebook checks the existing frozen Transformer scores against controlled SMART perturbations before designing an alert interface for the team's real-time collection agent.

Select clean August 15 **stop drives** from Notebook 17: these drives did not update encoder weights. Inject five simple changes into copies of their raw 30-day windows. Score original and changed windows with Notebook 18’s fixed checkpoint and thresholds; compare two SMART sector rules on the same windows. All transforms, thresholds and perturbation sizes are declared before scoring. No failure flags, evaluation drives, Q4, or sealed June are opened. Synthetic responsiveness is not recorded-failure recall or a reason to claim superiority.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader,TensorDataset
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
TRAIN_ID='q3_five_model_masked_transformer_v1'
SCORE_ID='q3_five_model_transformer_scoring_v1'
COMPARE_ID='q3_matched_detector_baseline_comparison_v1'
ABLATION_ID='q3_nuisance_controlled_regime_ablation_v1'
SENS_ID='q3_frozen_detector_synthetic_sensitivity_v1'
TRAIN=COMPONENT_ROOT/'results'/RUN_ID/TRAIN_ID
TRAINED=COMPONENT_ROOT/'models/trained'/RUN_ID/TRAIN_ID
SCORE=COMPONENT_ROOT/'results'/RUN_ID/SCORE_ID
COMPARE=COMPONENT_ROOT/'results'/RUN_ID/COMPARE_ID
ABLATION=COMPONENT_ROOT/'results'/RUN_ID/ABLATION_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/SENS_ID
OUT.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
trained=read_json(TRAIN/'encoder_training_complete.json')
scored=read_json(SCORE/'transformer_scoring_complete.json')
ablated=read_json(ABLATION/'nuisance_regime_ablation_complete.json')
assert trained['analysis_complete'] and scored['analysis_complete'] and ablated['analysis_complete']
assert not trained['failure_labels_used'] and not scored['failure_labels_used'] and not ablated['failure_labels_used']
assert not trained['future_test_outcomes_opened'] and not scored['future_test_opened'] and not ablated['future_test_opened']
assert not trained['sealed_June_test_opened'] and not scored['sealed_June_test_opened'] and not ablated['sealed_June_test_opened']
train_settings=read_json(TRAIN/'settings.json')
score_settings=read_json(SCORE/'settings.json')
prep_path=TRAINED/'preprocessing.json';prep=read_json(prep_path)
checkpoint_path=TRAINED/'best_model.pt';window_path=TRAIN/'fit_windows.npz'
assert sha256(prep_path)==trained['preprocessing_sha256']
assert sha256(checkpoint_path)==trained['best_checkpoint_sha256']==score_settings['checkpoint_sha256']
assert sha256(window_path)==trained['fit_windows_sha256']
assert sha256(SCORE/'calibrated_thresholds.json')==scored['thresholds_sha256']
assert sha256(COMPARE/'baseline_calibration_thresholds.json')==read_json(COMPARE/'score_lock.json')['baseline_threshold_sha256']
thresholds=read_json(SCORE/'calibrated_thresholds.json')
base_thresholds=read_json(COMPARE/'baseline_calibration_thresholds.json')
models=prep['models'];assert len(models)==5 and set(thresholds)==set(base_thresholds)==set(models)
FEATURES=prep['features']
assert FEATURES==['smart_5_raw','smart_9_raw','smart_194_raw','smart_197_raw','smart_198_raw']
assert score_settings['hidden_final_day_features']==['smart_5_raw','smart_194_raw','smart_197_raw','smart_198_raw']
assert score_settings['detectors']==['temp_error','sector_error','joint_error']
HIDDEN=[0,2,3,4]
SETTINGS={'checkpoint_sha256':sha256(checkpoint_path),'preprocessing_sha256':sha256(prep_path),
          'fit_windows_sha256':sha256(window_path),'transformer_threshold_sha256':sha256(SCORE/'calibrated_thresholds.json'),
          'baseline_threshold_sha256':sha256(COMPARE/'baseline_calibration_thresholds.json'),
          'ablation_complete_sha256':sha256(ABLATION/'nuisance_regime_ablation_complete.json'),
          'source':'August 15 stop drives with all 30 days of SMART 5,197,198 zero',
          'sample_per_model':200,'seed':42,
          'perturbations':['unchanged','temp_spike_final_8C','temp_shift_last7_6C',
                           'sector_onset_final','sector_ramp_last7','temp_flatline_last7'],
          'sector_onset':'final SMART 5=8, 197=2, 198=2',
          'sector_ramp':'last 7 days SMART 5=2,4,...14; 197 and 198=1 on final 4 days',
          'temp_flatline':'last 7 days SMART 194 repeats day -8',
          'threshold_rule':'strict greater than frozen per-model thresholds',
          'failure_labels_used':False,'winner_selection_enabled':False}
settings_path=OUT/'settings.json'
if settings_path.exists():assert read_json(settings_path)==SETTINGS,'Change SENS_ID before changing this protocol.'
else:settings_path.write_text(json.dumps(SETTINGS,indent=2))
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Checkpoint and frozen thresholds verified; device:',DEVICE)

Checkpoint and frozen thresholds verified; device: cpu


## 1. Select disjoint, sector-clean stopping windows and fix perturbations

Use the stored fit-window NPZ. Reconstruct Notebook 17’s stopping hash and sample 200 clean drives per model with a fixed seed. Copy arrays before modifying them. The perturbations are engineering probes, not realistic failure trajectories; report whether each probe actually changed the window.

In [2]:
with np.load(window_path) as a:
    windows=a['windows'];serials=a['serial_number'].astype(str);names=a['model'].astype(str)
    assert a['features'].tolist()==FEATURES
assert windows.shape==(len(serials),30,5) and len(set(serials))==len(serials)
def stop_flag(serial):
    h=hashlib.sha256((train_settings['stopping_salt']+'|'+str(serial)).encode()).hexdigest()[:16]
    return int(h,16)/16**16 < train_settings['stopping_fraction_of_fit_drives']
stop=np.array([stop_flag(s) for s in serials])
assert stop.sum()==trained['stopping_drives']
clean=(windows[:,:,[0,3,4]]==0).all(axis=(1,2))
chosen=[]
for model in models:
    candidates=np.flatnonzero(stop&clean&(names==model))
    assert len(candidates)>=SETTINGS['sample_per_model'],(model,len(candidates))
    rng=np.random.default_rng(SETTINGS['seed']+models.index(model))
    picked=np.sort(rng.choice(candidates,size=SETTINGS['sample_per_model'],replace=False))
    chosen.extend(picked.tolist())
chosen=np.array(chosen,dtype=int)
raw=windows[chosen].copy();ids=serials[chosen];model_names=names[chosen]
assert len(raw)==len(models)*SETTINGS['sample_per_model'] and np.isfinite(raw).all()
assert clean[chosen].all() and stop[chosen].all()
def inject(x,kind):
    v=x.copy()
    if kind=='unchanged':pass
    elif kind=='temp_spike_final_8C':v[:,-1,2]+=8
    elif kind=='temp_shift_last7_6C':v[:,-7:,2]+=6
    elif kind=='sector_onset_final':
        v[:,-1,0]=8;v[:,-1,3]=2;v[:,-1,4]=2
    elif kind=='sector_ramp_last7':
        v[:,-7:,0]=2*np.arange(1,8)[None,:]
        v[:,-4:,3]=1;v[:,-4:,4]=1
    elif kind=='temp_flatline_last7':v[:,-7:,2]=v[:,-8,2][:,None]
    else:raise ValueError(kind)
    assert np.isfinite(v).all() and (v>=0).all()
    return v
samples={kind:inject(raw,kind) for kind in SETTINGS['perturbations']}
assert np.array_equal(samples['unchanged'],raw)
for kind in SETTINGS['perturbations'][1:]:
    changed=np.any(samples[kind]!=raw,axis=(1,2))
    print(kind,'changed',int(changed.sum()),'of',len(changed))
assert all(np.any(samples[k]!=raw) for k in SETTINGS['perturbations'][1:])

temp_spike_final_8C changed 1000 of 1000
temp_shift_last7_6C changed 1000 of 1000
sector_onset_final changed 1000 of 1000
sector_ramp_last7 changed 1000 of 1000
temp_flatline_last7 changed 856 of 1000


## 2. Score paired original and perturbed windows with frozen methods

The checkpoint architecture and final-day mask exactly follow Notebook 18. Calculate the three reconstruction scores plus fixed/calibrated sector-positive and sector-delta baselines. The latter are included to verify obvious sector injections. Save row-level paired results and assert no hidden-target leakage through adjacent-day difference features.

In [3]:
class MaskedTelemetryTransformer(nn.Module):
    def __init__(self,n_models,width=64,heads=4,layers=2,feedforward=128,dropout=.1):
        super().__init__()
        self.input=nn.Linear(15,width)
        self.model_embedding=nn.Embedding(n_models,width)
        self.position=nn.Parameter(torch.zeros(1,30,width))
        nn.init.normal_(self.position,std=.02)
        block=nn.TransformerEncoderLayer(d_model=width,nhead=heads,dim_feedforward=feedforward,
                                          dropout=dropout,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(block,num_layers=layers,enable_nested_tensor=False)
        self.output=nn.Linear(width,5)
    def forward(self,x,model_id,hide):
        observed=(~hide).to(x.dtype)
        masked=x.masked_fill(hide,0)
        delta=torch.zeros_like(x)
        delta[:,1:,:]=x[:,1:,:]-x[:,:-1,:]
        delta_visible=observed.clone()
        delta_visible[:,1:,:]=observed[:,1:,:]*observed[:,:-1,:]
        delta=delta*delta_visible
        h=self.input(torch.cat([masked,observed,delta],dim=-1))
        h=h+self.model_embedding(model_id)[:,None,:]+self.position
        return self.output(self.encoder(h))
network=MaskedTelemetryTransformer(len(models),train_settings['width'],train_settings['heads'],
                                   train_settings['layers'],train_settings['feedforward'],train_settings['dropout']).to(DEVICE)
checkpoint=torch.load(checkpoint_path,map_location=DEVICE,weights_only=True)
assert checkpoint['models']==models and checkpoint['preprocessing_sha256']==sha256(prep_path)
network.load_state_dict(checkpoint['state_dict']);network.eval()
model_index={m:i for i,m in enumerate(models)}
@torch.no_grad()
def predict_errors(w,names):
    logged=np.log1p(w.astype(np.float64));scaled=np.empty_like(w,dtype=np.float32)
    for model in models:
        selected=names==model
        if selected.any():
            center=np.asarray(prep['center'][model],dtype=np.float64)
            scale=np.asarray(prep['scale'][model],dtype=np.float64)
            scaled[selected]=((logged[selected]-center)/scale).astype(np.float32)
    assert np.isfinite(scaled).all()
    model_ids=np.array([model_index[m] for m in names],dtype=np.int64)
    result=[]
    loader=DataLoader(TensorDataset(torch.from_numpy(scaled),torch.from_numpy(model_ids)),
                      batch_size=score_settings['batch_size'],shuffle=False,num_workers=0)
    for x,m in loader:
        x=x.to(DEVICE);m=m.to(DEVICE)
        hide=torch.zeros_like(x,dtype=torch.bool);hide[:,-1,HIDDEN]=True
        pred=network(x,m,hide)
        residual=(pred[:,-1,HIDDEN]-x[:,-1,HIDDEN]).square()
        result.append(residual.cpu().numpy())
    out=np.concatenate(result).astype(np.float32)
    assert out.shape==(len(w),4) and np.isfinite(out).all()
    return out
# Hidden target values cannot affect predictions, including through the delta channel.
probe=torch.from_numpy(np.log1p(raw[:2])).float().to(DEVICE)
m=torch.tensor([model_index[x] for x in model_names[:2]],device=DEVICE)
hide=torch.zeros_like(probe,dtype=torch.bool);hide[:,-1,HIDDEN]=True
with torch.no_grad():
    pred=network(probe,m,hide)
    changed=probe.clone();changed[hide]+=3
    torch.testing.assert_close(pred,network(changed,m,hide),rtol=1e-6,atol=1e-6)
rows=[]
for kind in SETTINGS['perturbations']:
    altered=samples[kind]
    errors=predict_errors(altered,model_names)
    table=pd.DataFrame({'serial_number':ids,'model':model_names,'scenario':kind,
                        'window_changed':np.any(altered!=raw,axis=(1,2)),
                        'temp_error':errors[:,1],
                        'sector_error':errors[:,[0,2,3]].mean(axis=1),
                        'joint_error':errors.mean(axis=1),
                        'sector_positive':altered[:,-1,[0,3,4]].max(axis=1),
                        'sector_delta':np.maximum(altered[:,-1,[0,3,4]]-altered[:,-2,[0,3,4]],0).sum(axis=1)})
    for name in ['temp_error','sector_error','joint_error']:
        table[name+'_flag']=table[name].to_numpy()>table.model.map(lambda x:thresholds[x][name]).to_numpy(dtype=float)
    for name in ['sector_positive','sector_delta']:
        table[name+'_fixed_flag']=table[name].to_numpy()>0
        table[name+'_calibrated_flag']=table[name].to_numpy()>table.model.map(lambda x:base_thresholds[x][name]).to_numpy(dtype=float)
    rows.append(table)
    print('Scored scenario:',kind)
scores=pd.concat(rows,ignore_index=True)
assert not scores.duplicated(['serial_number','model','scenario']).any()
score_path=OUT/'synthetic_paired_scores.parquet'
if score_path.exists():pd.testing.assert_frame_equal(pd.read_parquet(score_path),scores)
else:scores.to_parquet(score_path,index=False)
print('Paired synthetic score rows:',len(scores))

Scored scenario: unchanged
Scored scenario: temp_spike_final_8C
Scored scenario: temp_shift_last7_6C
Scored scenario: sector_onset_final
Scored scenario: sector_ramp_last7
Scored scenario: temp_flatline_last7
Paired synthetic score rows: 6000


## 3. Report incremental detection and alert burden on the stop-drive probe

For each method and perturbation, report the fraction of windows where the score rose, the raw alert rate, and **new alerts among windows unflagged at baseline**. Comparisons use paired drives. Counts are descriptive; the perturbed sample is deliberately balanced across models and is not a natural prevalence or false-alert estimate. The flatline case may produce no change when a real drive’s temperature was already constant.

In [4]:
methods=['temp_error','sector_error','joint_error','sector_positive_fixed','sector_delta_fixed',
         'sector_positive_calibrated','sector_delta_calibrated']
value_col={m:m if m in ['temp_error','sector_error','joint_error'] else ('sector_positive' if 'sector_positive' in m else 'sector_delta') for m in methods}
flag_col={m:(m+'_flag' if m in ['temp_error','sector_error','joint_error'] else m.replace('_fixed','').replace('_calibrated','')+('_fixed_flag' if m.endswith('_fixed') else '_calibrated_flag')) for m in methods}
baseline=scores.loc[scores.scenario.eq('unchanged')].set_index(['serial_number','model'])
rows=[]
for (model,scenario),group in scores.groupby(['model','scenario'],sort=False):
    idx=pd.MultiIndex.from_frame(group[['serial_number','model']])
    orig=baseline.loc[idx]
    assert len(group)==len(orig)==SETTINGS['sample_per_model']
    for method in methods:
        v=group[value_col[method]].to_numpy(dtype=float);prior=orig[value_col[method]].to_numpy(dtype=float)
        flags=group[flag_col[method]].to_numpy(dtype=bool);old=orig[flag_col[method]].to_numpy(dtype=bool)
        eligible=~old
        rows.append({'model':model,'scenario':scenario,'method':method,'sampled_drives':len(group),
                     'actually_changed_windows':int(group.window_changed.sum()),
                     'fraction_score_increased':float((v>prior+1e-8).mean()),
                     'median_score_change':float(np.median(v-prior)),
                     'baseline_alert_fraction':float(old.mean()),
                     'perturbed_alert_fraction':float(flags.mean()),
                     'unflagged_at_baseline':int(eligible.sum()),
                     'newly_flagged_count':int((flags&eligible).sum()),
                     'newly_flagged_fraction_of_initially_unflagged':float((flags&eligible).sum()/eligible.sum()) if eligible.any() else np.nan})
summary=pd.DataFrame(rows)
assert len(summary)==len(models)*len(SETTINGS['perturbations'])*len(methods)
assert summary.loc[summary.scenario.eq('unchanged'),'newly_flagged_count'].eq(0).all()
summary.to_csv(OUT/'controlled_sensitivity_summary.csv',index=False)
display(summary.loc[summary.scenario.ne('unchanged'),['model','scenario','method',
    'actually_changed_windows','newly_flagged_fraction_of_initially_unflagged',
    'perturbed_alert_fraction']])
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'settings_sha256':sha256(settings_path),'paired_scores_sha256':sha256(score_path),
        'summary_sha256':sha256(OUT/'controlled_sensitivity_summary.csv'),
        'failure_labels_used':False,'evaluation_drives_used':False,'future_test_opened':False,
        'sealed_June_test_opened':False,'winner_selected':False,
        'limitations':['injected perturbations are not natural HDD failure cases',
                       'only one August 15 stop window per sampled drive',
                       'calibration thresholds applied to a different stop-drive cohort',
                       'no inference about real failure recall or false-alert rate',
                       'regime and drift components not approved']}
(OUT/'synthetic_sensitivity_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 23 COMPLETE — frozen detector and SMART rules probed without labels.')
print('Next: inspect sensitivity and settle an operational candidate for agent integration; Q4 unopened.')

,model,scenario,method,actually_changed_windows,newly_flagged_fraction_of_initially_unflagged,perturbed_alert_fraction
35,TOSHIBA MG07ACA14TA,temp_spike_final_8C,temp_error,200,0.639594,0.630
36,TOSHIBA MG07ACA14TA,temp_spike_final_8C,sector_error,200,0.000000,0.000
37,TOSHIBA MG07ACA14TA,temp_spike_final_8C,joint_error,200,0.614213,0.605
38,TOSHIBA MG07ACA14TA,temp_spike_final_8C,sector_positive_fixed,200,0.000000,0.000
39,TOSHIBA MG07ACA14TA,temp_spike_final_8C,sector_delta_fixed,200,0.000000,0.000
...,...,...,...,...,...,...
205,HGST HMS5C4040BLE640,temp_flatline_last7,joint_error,186,0.000000,0.000
206,HGST HMS5C4040BLE640,temp_flatline_last7,sector_positive_fixed,186,0.000000,0.000
207,HGST HMS5C4040BLE640,temp_flatline_last7,sector_delta_fixed,186,0.000000,0.000
208,HGST HMS5C4040BLE640,temp_flatline_last7,sector_positive_calibrated,186,0.000000,0.000


NOTEBOOK 23 COMPLETE — frozen detector and SMART rules probed without labels.
Next: inspect sensitivity and settle an operational candidate for agent integration; Q4 unopened.
